## LangGraph

**LangGraph-> is a graph orchestration framework on the top the langchain, where your app is modeled as nodes(functions) and edges(transitions),with shared state flowing through.In real agent  for control flow we need branching, looping and retrying until the result gives success**

#### Part 1: **State-the heart of the langgraph**

In [19]:
from typing import TypedDict, Annotated
from operator import add

class AgentState(TypedDict):
    messages: Annotated[list,add] # add= reducer, it appends the messages not overwrite it
    question: str
    answer: str

#### Part-2: **Node**

In [20]:
def get_question(state:AgentState)-> AgentState: 
    return {"question":"What is langgraph"}

def answer_question(state:AgentState)-> AgentState:
    ans = f"Answer to :{state['question']}"
    return {"answer":ans}

#node function always return dict with only keys it wants to update - not the full state 

#### Part 3: **Building the state Graph**

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(AgentState)

builder.add_node("get_q",get_question)
builder.add_node("get_ans",answer_question)

builder.add_edge(START,"get_q")
builder.add_edge("get_q","get_ans") # edges are the transistion state for langgraph
builder.add_edge("get_ans",END)

graph = builder.compile()

for chunk in graph.stream({"messages":[], "question":"","answer":""},stream_mode="messages"):
# stream_mode options : "values" (full state), "updates" (only diffs), "messages" (token-by-token for chat)
    print(chunk)

#### Part 4: **Conditional Edges**

In [ ]:
# Conditional edges are branching the graph besed on the required contions in the graph
# first we have to create the function node with the conditions
def route_decision(state:AgentState):
    if "quantum" in state["question"]:
        return "quantum_node"
    else:
        return "normal_node"

builder.add_conditional_edges(
    "get_q", # source node
    route_decision, # function returning next node name 
    {
        "quantum_node":"quantum_node",
        "normal_node":"normal_node"
    }
    )

#### Part 5: **Loops**

In [ ]:
def check_quality(state:AgentState):
    if len(state["answer"]) < 20:
        return "retry"
    return "done"


builder.add_conditional_edges(
    "get_q",
    check_quality,
    {
        "retry" : "get_ans",  # loop back it self
        "done" : END
    }
    )

#### Part 6: **Checkpointing/Persistence (memory)**

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

def get_question(state:AgentState)-> AgentState: 
    return {"question":"What is langgraph"}

def answer_question(state:AgentState)-> AgentState:
    ans = f"Answer to :{state['question']}"
    return {"answer":ans}

builder = StateGraph(AgentState)

builder.add_node("get_q",get_question)
builder.add_node("get_ans",answer_question)

builder.add_edge(START,"get_q")
builder.add_edge("get_q","get_ans") # edges are the transistion state for langgraph
builder.add_edge("get_ans",END)

checkpointer = MemorySaver()  ## check pointer is hereeeeee

graph = builder.compile(checkpointer=checkpointer)

config = {"configurable":{"thread_id":"user-123"}}

# first call
graph.invoke({"messages":[{"role":"user","content":"Hi"}]},config=config)

# second call also can use the same config with thread-id for passing the old info
graph.invoke({"messages":[{"role":"user","content":"What did i say?"}]},config=config)

##### for production use **SqliteSaver** or **PostgreSaver**

In [ ]:
# from langgraph.checkpoint.sqlite import SqliteSaver

#checkpointer = SqliteSaver.from_conn_string("Checkpoint.db")

#### Part 7: **Human-in-the-loop**


In [ ]:
graph = builder.compile(
    interrupt_before=["get_ans"], # this intterept the execution before this node runs
    interrupt_after=["get_q"], # this intterept the execution after this node runs
    checkpointer=checkpointer
    )

In [ ]:
# you can also modify state manually before resuming:

graph.update_state(config,{"question":"modified question"})
graph.invoke(None,config=config)

#### Part 8: **Parallel Execution(Fan-In/Fan-out)**

In [ ]:
from langgraph.graph import START,END,StateGraph
from typing import TypedDict, Annotated

class ParallelState(TypedDict):
    topic:str
    result_a:str
    result_b:str
    summary: str

def node_a(state:ParallelState):
    return {"result_a":f"node_a analyse this {state['topic']}"}
def node_b(state:ParallelState):
    return {"result_b":f"node_a analyse this {state['topic']}"}
def merge(state:ParallelState):
    return {"summary":state["result_a"]+"+"+state["result_b"] }

builder_2 = StateGraph(ParallelState)

builder_2.add_node("a",node_a)
builder_2.add_node("b",node_b)
builder_2.add_node("merge",merge)

builder_2.add_edge(START,"a")
builder_2.add_edge(START,"b") # fan-out : both run parallel
builder_2.add_edge("a","merge")
builder_2.add_edge("b","merge") # fan-in: merge waits for both
builder_2.add_edge("merge",END)

graph = builder.compile()

# Langraph automatically detects a and b don't depend on each other and can run concurrently

#### Part 9: **Subgraphs**

In [ ]:
# You can nest acompiled graph inside another graph as a single node - useful for modular, reusable workflows
# Build and compile a smaller graph first
sub_builder = StateGraph(AgentState)
sub_builder.add_node("step1",get_question)
sub_builder.add_node("step2",answer_question)
sub_builder.add_edge(START,"step1")
sub_builder.add_edge("step1","step2")
sub_builder.add_edge("step2",END)
sub_graph = sub_builder.compile()

parent_builder = StateGraph(AgentState)
parent_builder.add_node("sub_workflow",sub_graph)
parent_builder.add_edge(START,"sub_workflow")
parent_builder.add_edge("sub_workflow",END)
parent_graph=parent_builder.compile()

#### Part 10: **Tool calling agent**

**This is the most common real-world LangGraph use cases: LLM decides to call tools, execute them, loop back with results util it has a final answer**

In [ ]:
from langgraph.graph import StateGraph,START,END,MessagesState
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from langchain_aws import ChatBedrock

@tool
def get_weather(city:str)-> str:
    """Get weather for city"""
    return f"{city} is sunny, 32 deg c"

tools =[get_weather]

llm = ChatBedrock(model_id ="enter your model id").blind_tools(tools)

def call_model(state:MessagesState):
    response = llm.invoke(state["messages"])
    return {"message":[response]}

def should_contine(state:MessagesState):
    last_msg = state["message"][-1]
    if last_msg.tool_calls: # LLM wants to call tools
        return "tools"
    return END


builder_llm = StateGraph(MessagesState)
builder_llm.add_node("agent",call_model)
builder_llm.add_node("tools",ToolNode(tools)) # prebuilt ToolNode using for this tool to add node
builder_llm.add_edge(START,"agent")
builder_llm.add_conditional_edges(
    "agent",
    should_contine,
    {
        "tools" : "tools",
         END: END

    }


)
builder_llm.add_edge("tools","agent")

graph = builder_llm.compile()

